 # Blotter Check & Integrity Verification Notebook

 Validates calculated blotter fields against underlying OHLCV and feature data,

 and performs step-by-step verification of agent observations against PyTorch model scalers.

In [1]:
import math
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.quant import QuantUtils
from core.settings import CacheConfig, TradingConfig, extract_run_hyperparameters
from data_pipeline.loader import load_processed_data

# Ensure project root is on sys.path dynamically
try:
    notebook_dir = Path(__file__).resolve().parent
except NameError:
    notebook_dir = Path.cwd()

if str(notebook_dir) not in sys.path:
    sys.path.append(str(notebook_dir))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

config = TradingConfig()


def toggle_quotes(text: str) -> str:
    """Auto-detects format and toggles quotes between quoted and unquoted strings."""
    is_quoted = '"' in text or "'" in text
    if is_quoted:
        return ", ".join(
            [item.strip().replace('"', "").replace("'", "") for item in text.split(",")]
        )
    else:
        return ", ".join([f'"{item.strip()}"' for item in text.split(",")])

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



 ### Section 2: Data Loading & Synchronized Run Auto-Discovery

In [2]:
# -----------------------------------------------------------------------------
# USER INPUT RUN SELECTION
# -----------------------------------------------------------------------------
# To audit a specific run, provide its filename, stem, or full path here:
# e.g., USER_SELECTED_FILE = "results_model_T3_Gamma090_Width5_Tilt85_s42_ent0.004_mult2.5_tilt0.85_w5_g0.9_rsc20.0_lr0.00015_ep_76_rollsh_0.87_valsh_0.78.pkl"
# If None or empty string (""), the notebook defaults to loading the latest model run.
USER_SELECTED_FILE: str | Path | None = (
    "results_model_T1_Convex_2x_Scale20_s42_ent0.0035_mult2.0_tilt0.7_w3_g0.9_rsc20.0_lr0.00015_ep_78_rollsh_1.09_valsh_1.08.pkl"
)
USER_SELECTED_FILE: str | Path | None = None

# 1. Load Data Pipelines & Reward Matrix
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df
alpha_df = pd.read_parquet(LOCAL_DATA_DIR / CacheConfig.get_filename())

# Pivot Close prices and compute 1-Day forward simple returns reward matrix
df_close = df_ohlcv["Adj Close"].unstack(level=0)
if config.handle_zeros_as_nan:
    df_close = df_close.replace(0.0, np.nan).ffill()
else:
    df_close = df_close.ffill(limit=config.max_data_gap_ffill)

reward_matrix = QuantUtils.build_forward_return_matrix(df_close, horizon=1)

# 2. Resolve Target Results Artifact (User Selected vs Default Auto-Discovery)
target_results_file = None
run_stem = ""

if USER_SELECTED_FILE:
    candidate_path = Path(USER_SELECTED_FILE)
    if candidate_path.is_file():
        selected_path = candidate_path
    elif (OUTPUT_DIR / USER_SELECTED_FILE).is_file():
        selected_path = OUTPUT_DIR / USER_SELECTED_FILE
    else:
        # Search by substring pattern
        search_pattern = f"*{USER_SELECTED_FILE}*"
        matches = list(OUTPUT_DIR.glob(search_pattern))
        if not matches:
            raise FileNotFoundError(
                f"❌ User-specified file or pattern '{USER_SELECTED_FILE}' not found in {OUTPUT_DIR}"
            )
        pkl_matches = [m for m in matches if m.suffix == ".pkl"]
        selected_path = pkl_matches[0] if pkl_matches else matches[0]

    print(f"🎯 Loading User-Selected Artifact: {selected_path.name}")

    if selected_path.suffix == ".parquet":
        run_stem = selected_path.stem.replace("blotter_df_", "")
        matching_pkl = OUTPUT_DIR / f"results_{run_stem}.pkl"
        target_results_file = matching_pkl if matching_pkl.exists() else None
    else:
        target_results_file = selected_path
        run_stem = selected_path.stem.replace("results_", "")
else:
    # Default behavior: discover latest evaluation results pickle
    results_files = list(OUTPUT_DIR.glob("results_*.pkl"))
    if not results_files:
        results_files = list(OUTPUT_DIR.glob("*.pkl"))
    if not results_files:
        raise FileNotFoundError(f"❌ No results pickle files found in {OUTPUT_DIR}")

    target_results_file = max(results_files, key=lambda p: p.stat().st_mtime)
    run_stem = target_results_file.stem.replace("results_", "")
    print(f"🚀 Loading Latest Results (Default): {target_results_file.name}")

results: dict = {}
if target_results_file and target_results_file.exists():
    try:
        with open(target_results_file, "rb") as f:
            results = pickle.load(f)
    except Exception as e:
        raise RuntimeError(f"❌ Failed to load pickle file {target_results_file}: {e}")

equity_curve = results.get("equity_curve")
alpha_equity_curve = results.get("alpha_equity_curve", "N/A")
metadata = results.get("metadata", {})

# 3. Synchronize Blotter from results or matching parquet
matching_parquet = OUTPUT_DIR / f"blotter_df_{run_stem}.parquet"

if matching_parquet.exists():
    print(f"🚀 Loading matching Blotter Parquet: {matching_parquet.name}")
    blotter_df = pd.read_parquet(matching_parquet)
    latest_blotter = matching_parquet
elif "blotter" in results:
    print(
        f"🚀 Extracting Blotter DataFrame directly from results['blotter'] ({target_results_file.name})"
    )
    blotter_df = pd.DataFrame(results["blotter"])
    matching_parquet.parent.mkdir(parents=True, exist_ok=True)
    blotter_df.to_parquet(matching_parquet, index=False)
    latest_blotter = matching_parquet
elif USER_SELECTED_FILE and Path(USER_SELECTED_FILE).suffix == ".parquet":
    print(f"🚀 Loading user-specified Blotter Parquet directly: {selected_path.name}")
    blotter_df = pd.read_parquet(selected_path)
    latest_blotter = selected_path
else:
    blotter_files = list(OUTPUT_DIR.glob("blotter_df_*.parquet"))
    if not blotter_files:
        raise FileNotFoundError(
            f"❌ No blotter data found for stem '{run_stem}' in {OUTPUT_DIR}"
        )
    latest_blotter = max(blotter_files, key=lambda p: p.stat().st_mtime)
    print(f"⚠️ Falling back to latest Blotter Parquet: {latest_blotter.name}")
    blotter_df = pd.read_parquet(latest_blotter)

# Ensure standardized datetime column exists
if "date" not in blotter_df.columns and "decision_date" in blotter_df.columns:
    blotter_df["date"] = pd.to_datetime(blotter_df["decision_date"])
elif "date" in blotter_df.columns:
    blotter_df["date"] = pd.to_datetime(blotter_df["date"])

artifact_filename = (
    target_results_file.name if target_results_file else latest_blotter.name
)
run_params = extract_run_hyperparameters(
    artifact_filename, metadata, config, blotter_df=blotter_df
)

print(
    f"\nTickers in df_ohlcv: {len(df_ohlcv.index.get_level_values('Ticker').unique())}"
)
print(f"Reward Matrix Shape: {reward_matrix.shape}")
print(f"Extracted Run Hyperparameters: {run_params}")

🚀 Loading Latest Results (Default): results_model_P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90.pkl
🚀 Loading matching Blotter Parquet: blotter_df_model_P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90.parquet

Tickers in df_ohlcv: 1708
Reward Matrix Shape: (7466, 1708)
Extracted Run Hyperparameters: {'min_basket_width': 5, 'min_width': 5, 'min_active_tilt': 0.1, 'upside_alpha_mult': 1.2, 'loss_aversion_penalty': 0.35, 'max_cash_pct': 0.0, 'slippage_rate': 0.001, 'holding_period': 5, 'gamma': 0.88, 'benchmark': 'SPY'}


 ### Section 3: Select Blotter Sample Record

In [3]:
blotter_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1109 entries, 0 to 1108
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   date                              1109 non-null   datetime64[ns]
 1   decision_date                     1109 non-null   object        
 2   buy_date                          1109 non-null   object        
 3   sell_date                         1109 non-null   object        
 4   universe_size                     1109 non-null   int64         
 5   max_score                         1109 non-null   float64       
 6   min_score                         1109 non-null   float64       
 7   observation                       1109 non-null   object        
 8   raw_actions                       1109 non-null   object        
 9   offset                            1109 non-null   int64         
 10  width                             1109 non-null 

In [4]:
# ==============================================================================
# CONFIGURABLE AUDIT SLICE SELECTION
# ==============================================================================
# Choose any slice index to audit:
# - Default: -1 (Terminal row / latest trading step)
# - Cold Start: 0 (Day 0 initialization)
# - Intermediate steps: e.g. 1, 42, 500, etc.
# Both positive and negative indexing are supported.
sample_idx = -1

# Universal row normalization to guarantee robust bounds handling
pos_idx = sample_idx % len(blotter_df)
blotter = blotter_df.iloc[pos_idx]


def get_field(row, *candidates, default="N/A"):
    for c in candidates:
        if c in row:
            val = row[c]
            if val is None:
                continue
            if isinstance(val, (list, tuple, np.ndarray)):
                if len(val) > 0:
                    return val
            elif pd.notna(val):
                return val
    return default


# Extract stored values from blotter supporting both modern and legacy column schemas
decision_date = get_field(blotter, "decision_date", "date")
buy_date = get_field(blotter, "buy_date")
sell_date = get_field(blotter, "sell_date")
universe_size = get_field(blotter, "universe_size")
max_score = get_field(blotter, "max_score")
min_score = get_field(blotter, "min_score")
observation = get_field(blotter, "observation", "obs", "state")
raw_actions = get_field(blotter, "raw_actions", "action", "actions")
decoded_offset = get_field(blotter, "offset", "decoded_offset")
decoded_width = get_field(blotter, "width", "decoded_width")
top_3_tickers = get_field(blotter, "top_3", "top_3_tickers")
chosen_tickers = get_field(
    blotter, "chosen_tickers", "selected", "selected_tickers", "tickers"
)
predicted_reward = get_field(blotter, "predicted_reward")
rl_reward_received = get_field(
    blotter,
    "rl_reward_received",
    "penalized_alpha_daily_simple_ret",
    "penalized_alpha",
)
raw_log_reward = get_field(
    blotter, "raw_log_reward", "penalized_alpha_daily_log_reward"
)
actual_return = get_field(
    blotter, "actual_return", "net_daily_simple_ret", "portfolio_impact"
)
mkt_return = get_field(blotter, "mkt_return", "bm_daily_simple_ret")
cash_return = get_field(blotter, "cash_return", "cash_daily_simple_ret", default=0.0)
alpha = get_field(blotter, "alpha", "alpha_daily_simple_ret")
penalized_alpha = get_field(
    blotter, "penalized_alpha", "penalized_alpha_daily_simple_ret"
)
slippage_applied = get_field(blotter, "slippage_applied", "slippage_daily_simple_loss")
portfolio_impact = get_field(
    blotter, "portfolio_impact", "net_daily_simple_ret", "actual_return"
)
alpha_impact = get_field(
    blotter, "alpha_impact", "penalized_alpha_daily_simple_ret", "penalized_alpha"
)
agent_equity = get_field(blotter, "agent_equity", "equity")
alpha_equity = get_field(blotter, "alpha_equity")

print(
    f"Selected Blotter Row [sample_idx = {sample_idx} -> pos_idx = {pos_idx} / {len(blotter_df) - 1}]:"
)
print(blotter)

Selected Blotter Row [sample_idx = -1 -> pos_idx = 1108 / 1108]:
date                                                              2026-09-02 00:00:00
decision_date                                                              2026-09-02
buy_date                                                                   2026-09-03
sell_date                                                                  2026-09-04
universe_size                                                                    1024
max_score                                                                    5.052856
min_score                                                                   -2.403981
observation                         [0.523531973361969, -1.0696030855178833, 0.030...
raw_actions                         [0.41301771998405457, -0.13293372094631195, -0...
offset                                                                            182
width                                                                      

In [5]:
def parse_tickers(val):
    """Safe parser for lists, ndarrays, and serialized strings."""
    if isinstance(val, (list, tuple, np.ndarray)):
        return list(val)
    elif isinstance(val, str):
        cleaned = (
            val.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        )
        return [t.strip() for t in cleaned.split(",") if t.strip()]
    return []


parsed_chosen = parse_tickers(chosen_tickers)
print("--- 1. Ticker Parsing Check ---")
print(f"Parsed chosen tickers: {parsed_chosen}")

print("\n--- 2. Date Alignment Check ---")
print(f"decision_date: {decision_date}")
print(f"buy_date:      {buy_date}")
print(f"sell_date:     {sell_date}")

--- 1. Ticker Parsing Check ---
Parsed chosen tickers: ['BKR', 'BMNR', 'VTRS', 'KDP', 'STRL', 'VEEV', 'FICO']

--- 2. Date Alignment Check ---
decision_date: 2026-09-02
buy_date:      2026-09-03
sell_date:     2026-09-04


 ### Section 4: Perform Step-by-Step Field Calculation Checks

In [6]:
from core.accounting import MTMPortfolioEngine
from core.logic import SelectionLogic

# Ensure decision_date is a Timestamp
decision_date_ts = pd.Timestamp(decision_date)

# 1. Verify Universe Size
alpha_result = alpha_df.loc[decision_date_ts]
cal_universe_size = len(alpha_result)

# 2. Decode Action via Stateless SelectionLogic Engine
raw_actions_arr = np.array(raw_actions)
(
    _,
    _,
    cal_decoded_offset,
    cal_decoded_width,
    cal_equity_exposure,
    cal_active_tilt,
    _,
    _,
) = SelectionLogic.apply_action(
    ensemble=alpha_result,
    action=raw_actions_arr,
    rank_max_offset_percentile=config.rank_max_offset_percentile,
    rank_max_width=config.rank_max_width,
    min_basket_width=run_params["min_basket_width"],
    max_cash_pct=run_params["max_cash_pct"],
    min_active_tilt=run_params["min_active_tilt"],
)

# 3. Verify 1-Day Benchmark & Cash Returns
benchmark_ticker = getattr(
    config, "benchmark", getattr(config, "benchmark_ticker", "SPY")
)
if (
    decision_date_ts in reward_matrix.index
    and benchmark_ticker in reward_matrix.columns
):
    bm_val = reward_matrix.loc[decision_date_ts, benchmark_ticker]
    bm_raw = float(np.asarray(bm_val).ravel()[0])
    cal_mkt_return = float(bm_raw) if np.isfinite(bm_raw) else 0.0
else:
    cal_mkt_return = float(
        blotter.get("bm_daily_simple_ret", blotter.get("mkt_return", 0.0))
    )

cal_cash_return = float(
    blotter.get("cash_daily_simple_ret", blotter.get("cash_return", 0.0))
)

# 4. Reconstruct Rolling FIFO Active Sleeves with Strict T+1 Close Execution
# Under T+1 Close convention:
# - Decision made on Day T is pending over (T -> T+1) with ZERO risk/PnL exposure.
# - At Day T+1 (Close), pending sleeve executes and joins active sleeves.
# - At step `pos_idx`, the executing sleeve is the pending order from step `pos_idx - 1`.
# - Active sleeves earning return over (pos_idx -> pos_idx + 1) are decisions from:
#   [max(0, pos_idx - holding_period) : pos_idx] (exclusive of pos_idx).
holding_period = run_params["holding_period"]
tickers_col = next(
    (
        c
        for c in ["chosen_tickers", "selected", "selected_tickers", "tickers"]
        if c in blotter_df.columns
    ),
    "chosen_tickers",
)

if pos_idx == 0:
    # Day 0 (Cold Start): Zero pending sleeve executed; zero active sleeves.
    executing_sleeve = None
    active_sleeves_history = []
else:
    # Day pos_idx: Pending sleeve from pos_idx - 1 executes NOW at Close.
    executing_sleeve = parse_tickers(blotter_df.iloc[pos_idx - 1][tickers_col])
    active_start_idx = max(0, pos_idx - holding_period)
    active_sleeves_history = blotter_df.iloc[active_start_idx:pos_idx][
        tickers_col
    ].tolist()

# 5. Verify 1-Day MTM Active Stock Return using forward reward_matrix
# Capital is partitioned into H equal portions (1/H weight per sleeve).
# Unfilled sleeves (H - num_active) hold benchmark.
num_active = len(active_sleeves_history)
if num_active > 0:
    sleeve_daily_returns = []
    for sleeve_tickers in active_sleeves_history:
        chosen_list = parse_tickers(sleeve_tickers)
        if len(chosen_list) > 0 and decision_date_ts in reward_matrix.index:
            sleeve_ret = MTMPortfolioEngine.calculate_sleeve_simple_ret(
                chosen_list, reward_matrix.loc[decision_date_ts]
            )
            sleeve_daily_returns.append(sleeve_ret)
        else:
            sleeve_daily_returns.append(0.0)

    active_sum = float(np.sum(sleeve_daily_returns))
    unfilled_count = holding_period - num_active
    cal_raw_stock_return = (
        active_sum + (unfilled_count * cal_mkt_return)
    ) / holding_period
else:
    # During cold start (Day 0 -> Day 1), 0 sleeves active -> 100% benchmark return
    cal_raw_stock_return = cal_mkt_return

# 6. Verify Tri-Asset Allocation Weights
current_chosen = parse_tickers(chosen_tickers)
if num_active > 0 or len(current_chosen) > 0:
    cal_w_active = cal_equity_exposure * cal_active_tilt
    cal_w_benchmark = cal_equity_exposure * (1.0 - cal_active_tilt)
    cal_w_cash = max(0.0, 1.0 - (cal_w_active + cal_w_benchmark))
else:
    cal_w_active = 0.0
    cal_w_benchmark = cal_equity_exposure
    cal_w_cash = max(0.0, 1.0 - (cal_w_active + cal_w_benchmark))

# 7. Verify Gross Return, Amortized Slippage & Net Actual Return
cal_gross_return = (
    cal_w_active * cal_raw_stock_return
    + cal_w_benchmark * cal_mkt_return
    + cal_w_cash * cal_cash_return
)

slippage_rate = float(
    getattr(config, "slippage_rate", run_params.get("slippage_rate", 0.0010))
)
cal_slippage_applied = (
    float((slippage_rate / holding_period) * cal_w_active)
    if executing_sleeve is not None and len(executing_sleeve) > 0 and cal_w_active > 0.0
    else 0.0
)

cal_actual_return = cal_gross_return - cal_slippage_applied
cal_alpha = cal_actual_return - cal_mkt_return

# 8. Verify Penalized Alpha / RL Reward
upside_mult = run_params["upside_alpha_mult"]
loss_aversion_penalty = run_params["loss_aversion_penalty"]

if cal_alpha > 0.0:
    cal_penalized_alpha = cal_alpha * upside_mult
elif cal_alpha < 0.0:
    loss_multiplier = 1.0 + max(0.0, loss_aversion_penalty)
    cal_penalized_alpha = cal_alpha * loss_multiplier
else:
    cal_penalized_alpha = 0.0

if not np.isfinite(cal_penalized_alpha):
    cal_penalized_alpha = 0.0

cal_rl_reward_received = float(cal_penalized_alpha)

# 9. Verify 1-Day Portfolio & Alpha Impacts
cal_portfolio_impact = cal_actual_return
cal_alpha_impact = cal_penalized_alpha

############################

# 10. Verify Compounded Equity Curves
# agent_eq_col = next(
#     (
#         c
#         for c in ["agent_equity", "equity", "portfolio_equity"]
#         if c in blotter_df.columns
#     ),
#     "agent_equity",
# )
# bm_eq_col = next(
#     (c for c in ["bm_equity", "benchmark_equity"] if c in blotter_df.columns),
#     "bm_equity",
# )
# alpha_eq_col = next(
#     (c for c in ["alpha_equity"] if c in blotter_df.columns), "alpha_equity"
# )

# if pos_idx == 0:
#     prev_agent_equity = 1.0
#     prev_benchmark_equity = 1.0
# else:
#     prev_agent_equity = float(blotter_df.iloc[pos_idx - 1][agent_eq_col])
#     prev_benchmark_equity = float(blotter_df.iloc[pos_idx - 1][bm_eq_col])


# 10. Verify Compounded Equity Curves using canonical blotter schema
if pos_idx == 0:
    prev_agent_equity = 1.0
    prev_benchmark_equity = 1.0
else:
    prev_agent_equity = float(blotter_df.iloc[pos_idx - 1]["agent_equity"])
    prev_benchmark_equity = float(blotter_df.iloc[pos_idx - 1]["bm_equity"])

############################

# Strict Geometric Wealth Ratio (Base 1.0: V_p(t) / V_bm(t))
cal_agent_equity, cal_bm_equity, cal_alpha_equity = QuantUtils.step_compounding_curves(
    prev_portfolio_equity=prev_agent_equity,
    prev_benchmark_equity=prev_benchmark_equity,
    net_daily_ret=cal_actual_return,
    bm_daily_ret=cal_mkt_return,
)

 ### Section 5: Verification Dashboard & Match Report

In [7]:
verification_records = [
    ("universe_size", universe_size, cal_universe_size),
    ("mkt_return", mkt_return, cal_mkt_return),
    ("actual_return", actual_return, cal_actual_return),
    ("alpha", alpha, cal_alpha),
    ("penalized_alpha", penalized_alpha, cal_penalized_alpha),
    ("rl_reward_received", rl_reward_received, cal_rl_reward_received),
    ("slippage_applied", slippage_applied, cal_slippage_applied),
    ("portfolio_impact", portfolio_impact, cal_portfolio_impact),
    ("alpha_impact", alpha_impact, cal_alpha_impact),
    ("agent_equity", agent_equity, cal_agent_equity),
    ("alpha_equity", alpha_equity, cal_alpha_equity),
    ("decoded_offset", decoded_offset, cal_decoded_offset),
    ("decoded_width", decoded_width, cal_decoded_width),
]

summary_df = pd.DataFrame(
    verification_records,
    columns=["Metric", "Blotter Value", "Calculated Value"],
)

# Float precision check (1e-6 tolerance)
summary_df["Match"] = summary_df.apply(
    lambda row: math.isclose(
        float(row["Blotter Value"]),
        float(row["Calculated Value"]),
        abs_tol=1e-6,
    ),
    axis=1,
)

summary_df["Status"] = summary_df["Match"].map({True: "✅ PASS", False: "❌ FAIL"})

pd.set_option("display.float_format", lambda x: f"{x:.8f}")

print("======================================================================")
print(
    f" DAILY MTM BLOTTER VERIFICATION SUMMARY (sample_idx={sample_idx}, pos_idx={pos_idx}, H={holding_period})"
)
print("======================================================================")
print(summary_df[["Metric", "Blotter Value", "Calculated Value", "Status"]])
print("======================================================================")

if summary_df["Match"].all():
    print("🎉 ALL DAILY MTM METRICS VERIFIED SUCCESSFULLY!")
else:
    failed_metrics = summary_df[~summary_df["Match"]]["Metric"].tolist()
    print(f"⚠️ VERIFICATION FAILED FOR METRICS: {failed_metrics}")

 DAILY MTM BLOTTER VERIFICATION SUMMARY (sample_idx=-1, pos_idx=1108, H=5)
                Metric  Blotter Value  Calculated Value  Status
0        universe_size  1024.00000000     1024.00000000  ✅ PASS
1           mkt_return     0.01046840        0.01046840  ✅ PASS
2        actual_return     0.01043112        0.01043112  ✅ PASS
3                alpha    -0.00003728       -0.00003728  ✅ PASS
4      penalized_alpha    -0.00005033       -0.00005033  ✅ PASS
5   rl_reward_received    -0.00005033       -0.00005033  ✅ PASS
6     slippage_applied     0.00018850        0.00018850  ✅ PASS
7     portfolio_impact     0.01043112        0.01043112  ✅ PASS
8         alpha_impact    -0.00005033       -0.00005033  ✅ PASS
9         agent_equity     1.46913533        1.46913533  ✅ PASS
10        alpha_equity     0.81228361        0.81228361  ✅ PASS
11      decoded_offset   182.00000000      182.00000000  ✅ PASS
12       decoded_width     7.00000000        7.00000000  ✅ PASS
🎉 ALL DAILY MTM METRICS VERIF

 ### Section 6: Independent Observation Verification Engine

In [8]:
def verify_agent_observation(
    blotter_source: Path | pd.DataFrame,
    pt_model_path: Path,
    target_date: pd.Timestamp,
):
    """Reconstructs 46-dim agent observations manually from raw cache & scaler state and verifies against recorded blotter values."""
    date_str = target_date.strftime("%Y-%m-%d")
    print(f"\n🔍 Starting independent verification for date: {date_str}")

    # --- 1. LOAD RAW PIPELINE & CACHE ---
    print("   -> Loading Macro DataFrame...")
    data = load_processed_data()
    df_ohlcv = data.df_ohlcv
    macro_df = data.macro_df
    features_df = data.features_df

    print("   -> Loading AlphaCache (feature_cube)...")
    cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
    feature_cube = pd.read_parquet(cache_file_path)

    # --- 2. LOAD BLOTTER ---
    if isinstance(blotter_source, pd.DataFrame):
        df_target = blotter_source.copy()
    else:
        print(f"   -> Loading Blotter: {blotter_source.name}")
        df_target = pd.read_parquet(blotter_source)

    date_col = next(
        (c for c in ["date", "decision_date"] if c in df_target.columns), None
    )
    if not date_col:
        raise KeyError("❌ No date/decision_date column found in blotter_df.")

    df_target[date_col] = pd.to_datetime(df_target[date_col])

    obs_col = next(
        (
            col
            for col in ["obs", "observation", "state", "raw_obs"]
            if col in df_target.columns
        ),
        None,
    )
    if not obs_col:
        raise KeyError("❌ Could not find an observation column in blotter_df.")

    blotter_row = df_target[df_target[date_col] == target_date]
    if blotter_row.empty:
        raise ValueError(f"❌ Date {date_str} not found in blotter_df!")
    blotter_row = blotter_row.iloc[0]
    blotter_obs = np.array(blotter_row[obs_col])

    # --- 3. LOAD SCALER WEIGHTS ---
    print(f"   -> Extracting Scaler State from: {pt_model_path.name}")
    checkpoint = torch.load(pt_model_path, map_location="cpu", weights_only=False)
    scaler_state = checkpoint.get("scaler_state")
    if not scaler_state:
        raise ValueError(
            "❌ 'scaler_state' not found inside the .pt checkpoint dictionary."
        )

    # --- 4. RECONSTRUCT 46-DIM OBSERVATION MANUALLY ---
    print("\n⚙️ Reconstructing 46-Dim Observation Manually...")
    expected_strats = len(feature_cube.columns)

    # STEP A: Cross-Sectional Universe Mean & Std (Dims 0..23)
    if target_date in feature_cube.index.get_level_values("Date"):
        ensemble = feature_cube.xs(target_date, level="Date")
    else:
        ensemble = pd.DataFrame()

    if not ensemble.empty and ensemble.shape[1] == expected_strats:
        strat_mean = np.nan_to_num(
            np.asarray(ensemble.mean(axis=0), dtype=float), nan=0.0
        )
        strat_std = np.nan_to_num(
            np.asarray(ensemble.std(axis=0, ddof=0), dtype=float), nan=0.0
        )
    else:
        strat_mean = np.zeros(expected_strats)
        strat_std = np.zeros(expected_strats)

    # STEP B: Benchmark Feature Vector Slice (Dims 24..35)
    benchmark = getattr(config, "benchmark", getattr(config, "benchmark_ticker", "SPY"))
    try:
        bm_cube = feature_cube.xs(benchmark, level="Ticker")
        if target_date in bm_cube.index:
            bm_vals = bm_cube.loc[target_date].fillna(0.0).values
        else:
            bm_vals = np.zeros(expected_strats)
    except KeyError:
        bm_vals = np.zeros(expected_strats)

    # STEP C: Stationary Macro Context (Dims 36..45)
    if target_date in macro_df.index:
        macro_row = macro_df.loc[target_date]
    else:
        macro_row = pd.Series(0.0, index=macro_df.columns)
    macro_vals = macro_row.fillna(0.0).values

    # Concatenate: 12 Mean + 12 Std + 12 Benchmark + 10 Macro = 46 Dims
    raw_obs = np.concatenate(
        [
            np.asarray(strat_mean),
            np.asarray(strat_std),
            np.asarray(bm_vals),
            np.asarray(macro_vals),
        ]
    ).astype(np.float32)

    raw_obs = np.nan_to_num(raw_obs, nan=0.0, posinf=0.0, neginf=0.0)

    # STEP D: Apply ObservationScaler.transform()
    mean = scaler_state["mean"]
    var = scaler_state["var"]
    count = scaler_state["count"]

    variance = var / count
    std = np.sqrt(variance) + 1e-8
    scaled_x = (raw_obs - mean) / std
    manual_final_obs = np.arcsinh(scaled_x)

    # --- 5. AUDIT & COMPARISON ---
    diff = np.abs(manual_final_obs - blotter_obs)
    max_diff = np.max(diff)

    print("\n" + "=" * 50)
    print("📊 VERIFICATION RESULTS")
    print("=" * 50)
    print(
        f"Observation Dimension: {len(manual_final_obs)} (Blotter: {len(blotter_obs)})"
    )
    print(f"Max Absolute Error:    {max_diff:.8f}")

    if max_diff < 1e-5:
        print("✅ SUCCESS: Independent math perfectly matches the Blotter observation!")
    else:
        print("❌ FAILED: Drift detected between manual calculation and blotter.")

    print("\n--- Snippet Comparison (First 5 Dims) ---")
    print(f"{'Dim':<5} | {'Manual Calculation':<20} | {'Blotter Recording'}")
    print("-" * 50)
    for i in range(5):
        print(f"{i:<5} | {manual_final_obs[i]:<20.6f} | {blotter_obs[i]:.6f}")

 ### Section 7: Dynamic Observation Verification Execution

In [9]:
# 1. Match model checkpoint corresponding to the loaded run
checkpoint_dir = OUTPUT_DIR / "model_checkpoints"

candidate_names = [
    f"{run_stem}.pt",
    f"{run_stem}.pth",
]
pt_model_path = None

for c_name in candidate_names:
    if (checkpoint_dir / c_name).exists():
        pt_model_path = checkpoint_dir / c_name
        break
    elif (OUTPUT_DIR / c_name).exists():
        pt_model_path = OUTPUT_DIR / c_name
        break

# Fallback: search by matching stem or latest checkpoint
if pt_model_path is None:
    matching_pts = list(checkpoint_dir.glob(f"*{run_stem}*.pt")) or list(
        OUTPUT_DIR.glob(f"*{run_stem}*.pt")
    )
    if matching_pts:
        pt_model_path = max(matching_pts, key=lambda p: p.stat().st_mtime)
    else:
        all_pts = list(checkpoint_dir.glob("*.pt")) or list(OUTPUT_DIR.glob("*.pt"))
        if all_pts:
            pt_model_path = max(all_pts, key=lambda p: p.stat().st_mtime)
        else:
            raise FileNotFoundError(
                f"❌ Cannot find checkpoint matching run '{run_stem}'"
            )

print(f"🧠 Matched PyTorch Model: {pt_model_path.name}")

# 2. Dynamic target date aligned directly with the audited slice date
target_date = pd.Timestamp(decision_date)
print(
    f"🎯 Verifying agent observation for slice date: {target_date.strftime('%Y-%m-%d')} (pos_idx={pos_idx})"
)

# 3. Execute observation verification
verify_agent_observation(
    blotter_source=blotter_df,
    pt_model_path=pt_model_path,
    target_date=target_date,
)

🧠 Matched PyTorch Model: model_P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90.pt
🎯 Verifying agent observation for slice date: 2026-09-02 (pos_idx=1108)

🔍 Starting independent verification for date: 2026-09-02
   -> Loading Macro DataFrame...
   -> Loading AlphaCache (feature_cube)...
   -> Extracting Scaler State from: model_P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5_s42_ep_90.pt

⚙️ Reconstructing 46-Dim Observation Manually...

📊 VERIFICATION RESULTS
Observation Dimension: 46 (Blotter: 46)
Max Absolute Error:    0.00000000
✅ SUCCESS: Independent math perfectly matches the Blotter observation!

--- Snippet Comparison (First 5 Dims) ---
Dim   | Manual Calculation   | Blotter Recording
--------------------------------------------------
0     | 0.523532             | 0.523532
1     | -1.069603            | -1.069603
2     | 0.030626             | 0.030626
3     | 0.027116             | 0.027116
4     | 0.402186             | 0.402186
